In [ ]:
import pandas as pd
import re
import os
import numpy as np
from random import random

os.environ["WANDB_DISABLED"] = "true"

from google.colab import drive
from sklearn.model_selection import KFold
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sentence_transformers import SentenceTransformer, util

# ================================
# Step 3: Mount Google Drive
# ================================
drive.mount('/content/drive')

# ================================
# Step 4: Verify file existence
# ================================
#!ls "/content/drive/MyDrive/bertmixeddataset/"

# ================================
# Step 5: Load the dataset
# ================================
file_path = '/content/drive/MyDrive/data2/mixeddataset.xlsx'
df = pd.read_excel(file_path)
df = pd.read_excel(file_path)

# 🔥 CLEAN COLUMN NAMES (MANDATORY)
df.columns = (
    df.columns
      .astype(str)
      .str.strip()
      .str.lower()
      .str.replace('\ufeff', '', regex=False)
)

print("Columns after cleaning:", df.columns.tolist())

print("✅ File loaded successfully!")

# ================================
# Step 6: Preprocess the dataset
# ================================
def preprocess_text(text):
    if isinstance(text, str):
        text = re.sub(r'[^\u0000-\u007F]+', ' ', text) # Remove non-ASCII characters
        text = re.sub(r'\s+', ' ', text).strip()
        return ' '.join(text.split()[:500]) # Truncate to first 500 words
    return ''

# Rename columns to 'subject', 'body', and 'label' for consistency
df.rename(columns={
    'Subject': 'subject',
    'Body': 'body',
    'Label': 'label'
}, inplace=True)

df['subject'] = df['subject'].apply(preprocess_text)
df['body'] = df['body'].apply(preprocess_text)

df['text'] = df['subject'] + ' ' + df['body']
df['label'] = df['label'].astype(int)

df = df.dropna(subset=['text', 'label'])
df = df[df['text'].str.strip() != '']
df = df[~df['text'].str.contains('#ERROR!', na=False)]

# ================================
# Add label noise (5%)
# ================================
def add_label_noise(label, noise_rate=0.05):
    return 1 - label if random() < noise_rate else label

df['label'] = df['label'].apply(add_label_noise)

# ================================
# Remove exact duplicates
# ================================
duplicates = df.duplicated(subset=['text']).sum()
print(f"Exact duplicates found: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates(subset=['text'])

# ================================
# Remove near-duplicates
# ================================
print("Checking for near-duplicates...")

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(df['text'].tolist(), batch_size=32, show_progress_bar=True)

cos_sim = util.cos_sim(embeddings, embeddings)
threshold = 0.90

near_duplicates = set()
for i in range(len(cos_sim)):
    for j in range(i + 1, len(cos_sim)):
        if cos_sim[i][j] > threshold:
            near_duplicates.add(j)

print(f"Near-duplicates removed: {len(near_duplicates)}")

if near_duplicates:
    df = df.drop(index=df.index[list(near_duplicates)])

# ================================
# Step 7: Tokenizer
# ================================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# ================================
# Step 8: Metrics
# ================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

# ================================
# Step 9: 5-Fold Cross Validation
# ================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
    print(f"\n===== Training Fold {fold + 1}/5 =====")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
    val_dataset = Dataset.from_pandas(val_df[['text', 'label']])

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)

    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

    model.config.hidden_dropout_prob = 0.5
    model.config.attention_probs_dropout_prob = 0.5

    for param in model.bert.encoder.layer[:6].parameters():
        param.requires_grad = False

    training_args = TrainingArguments(
        output_dir=f'/content/drive/MyDrive/bertmixeddataset/bertfolder/fold_{fold + 1}',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=100,
        weight_decay=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_strategy="steps",
        logging_steps=50,
        logging_first_step=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    fold_metrics.append(trainer.evaluate())

# ================================
# Step 10: Final training
# ================================
dataset = Dataset.from_pandas(df[['text', 'label']])
split = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split['train']
val_dataset = split['test']

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

model.config.hidden_dropout_prob = 0.5
model.config.attention_probs_dropout_prob = 0.5

for param in model.bert.encoder.layer[:6].parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/bertmixeddataset/bertfolder',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

# ================================
# Step 11: Save final model
# ================================
model.save_pretrained('/content/drive/MyDrive/bertmixeddataset/bertfolder')
tokenizer.save_pretrained('/content/drive/MyDrive/bertmixeddataset/bertfolder')

print("\n✅ Training complete. BERT model saved ")

In [ ]:
# ============================================
# STEP 1: Mount Google Drive (IMPORTANT)
# ============================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# (Optional but recommended) Check model files
!ls /content/drive/MyDrive/bertmixeddataset/bertfolder


# ============================================
# STEP 2: Import required libraries
# ============================================
from transformers import BertTokenizer, BertForSequenceClassification
import torch


# ============================================
# STEP 3: Load fine-tuned BERT model (LOCAL ONLY)
# ============================================
model_path = "/content/drive/MyDrive/bertmixeddataset/bertfolder"

tokenizer = BertTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

model = BertForSequenceClassification.from_pretrained(
    model_path,
    local_files_only=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("✅ BERT model loaded successfully")
print("🖥️ Device:", device)
print("Type 'exit' as subject to stop testing\n")


# ============================================
# STEP 4: Interactive email testing loop
# ============================================
while True:
    subject = input("📧 Enter email subject: ")

    if subject.lower() == "exit":
        print("\n🛑 Testing stopped")
        break

    body = input("📝 Enter email body: ")

    # Combine subject + body
    text = subject + " " + body

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Prediction
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred].item()

    label = "🚨 Phishing Email" if pred == 1 else "✅ Legitimate Email"

    # Output
    print("\n🔍 Prediction:", label)
    print("📊 Confidence:", round(confidence * 100, 2), "%\n")
